# Hybrid BML Segmentation: InceptionResNetV2-UNet with Patch Feature Fusion

This notebook implements a hybrid deep learning segmentation model for detecting
**Bone Marrow Lesions (BML)** in MRI images.

## Architecture Overview
- **Global encoder**: InceptionResNetV2 processes the full 448×448 image and exposes skip connections at 5 spatial scales.
- **Patch encoder**: Each image is split into a 4×4 grid of 112×112 patches; every patch is independently encoded by a second IRv2 instance, and the 7×7 patch bottleneck features are folded back into a 28×28 spatial map.
- **Feature fusion**: The global and patch bottleneck features are concatenated and refined with a Squeeze-and-Excitation (SE) recalibration block.
- **Decoder**: Four progressive upsampling blocks restore the full 448×448 resolution.
- **Output**: Binary segmentation mask predicted via sigmoid activation.

## Requirements
```
tensorflow >= 2.8
keras
classification-models (qubvel/classification_models)
opencv-python
scikit-image
scikit-learn
tqdm
matplotlib
pandas
```

In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.optimizers import AdamW

# XLA requires the CUDA directory — propagate from CUDA_HOME if set on your system
os.environ['CUDA_DIR'] = os.environ['CUDA_HOME']
os.environ["XLA_FLAGS"] = "--xla_gpu_cuda_data_dir=" + os.environ["CUDA_HOME"]

# Make all 4 GPUs visible — adjust device IDs to match your hardware
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"

gpus = tf.config.list_physical_devices("GPU")
print("Physical GPUs:", len(gpus), gpus)

# Enable memory growth to avoid pre-allocating all GPU memory at startup
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

# MirroredStrategy distributes training synchronously across all visible GPUs
strategy = tf.distribute.MirroredStrategy()
print("Replicas:", strategy.num_replicas_in_sync)

# Disable layout optimizer — can conflict with custom layer output shapes
tf.config.optimizer.set_experimental_options({"layout_optimizer": False})

print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))

In [ ]:
# ─── Standard library ───────────────────────────────────────────────────────
import os
import sys

# ─── Numerical / vision stack ────────────────────────────────────────────────
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from skimage.transform import resize
from sklearn.model_selection import train_test_split, KFold

# ─── TensorFlow / Keras ──────────────────────────────────────────────────────
import keras
import tensorflow as tf
from tensorflow.keras import backend as K, Model
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import Adam, AdamW
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.losses import binary_crossentropy
from tensorflow.keras.regularizers import l2
from tensorflow.keras.utils import plot_model, get_file
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import InceptionResNetV2, EfficientNetB0, EfficientNetV2B0
from tensorflow.keras.applications.inception_resnet_v2 import preprocess_input
from tensorflow.keras.layers import (
    Input, Conv2D, Conv2DTranspose, MaxPooling2D, UpSampling2D,
    Dropout, BatchNormalization, Activation, SpatialDropout2D,
    Concatenate, add, concatenate, ZeroPadding2D, AveragePooling2D,
    GlobalAveragePooling2D, GlobalMaxPooling2D, Lambda,
    Dense, Layer, Softmax, Reshape, Multiply, Add, ReLU
)

# ─── Third-party classification model zoo ────────────────────────────────────
from classification_models.keras import Classifiers
from classification_models.tfkeras import Classifiers as TFClassifiers

ResNet18, _ = Classifiers.get('resnet18')

# ═════════════════════════════════════════════════════════════════════════════
# Dataset & output paths
# TODO: Replace these placeholder paths with your actual dataset location.
#
#   Expected directory structure:
#       <dataset_root>/Train/images/    — training MRI images (.bmp)
#       <dataset_root>/Train/masks/     — corresponding binary masks (.bmp)
#       <dataset_root>/Validate/images/
#       <dataset_root>/Validate/masks/
#       <dataset_root>/Test/images/
#       <dataset_root>/Test/masks/
# ═════════════════════════════════════════════════════════════════════════════
traingdata = "<PATH_TO_YOUR_DATASET>/Train/images"
traingmask  = "<PATH_TO_YOUR_DATASET>/Train/masks"

validdata  = "<PATH_TO_YOUR_DATASET>/Validate/images"
validmask   = "<PATH_TO_YOUR_DATASET>/Validate/masks"

testingdata = "<PATH_TO_YOUR_DATASET>/Test/images"
testingmask = "<PATH_TO_YOUR_DATASET>/Test/masks"

# TODO: Update to your preferred output directories
WEIGHTS_DIR    = "<PATH_TO_YOUR_OUTPUT>/weights"
PREDICTION_DIR = "<PATH_TO_YOUR_OUTPUT>/predictions"

# ─── Training hyper-parameters ────────────────────────────────────────────────
width      = 448
height     = 448
EPOCHS     = 100
BATCH_SIZE = 32
num_folds  = 3
fold_no    = 1

# TODO: Update to your preferred model checkpoint path
modelname = "<PATH_TO_YOUR_OUTPUT>/InceptionResNetV2_BML.h5"

seed = 7
np.random.seed(seed)

# ═════════════════════════════════════════════════════════════════════════════
# U-Net building blocks
# ═════════════════════════════════════════════════════════════════════════════

def conv_block(input, num_filters):
    """Two consecutive Conv→BN→ReLU layers (standard U-Net double-conv)."""
    x = Conv2D(num_filters, 3, padding="same")(input)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    x = Conv2D(num_filters, 3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    return x

def decoder_block(input, skip_features, num_filters):
    """Transposed-conv upsample → concat skip connection → double-conv."""
    x = Conv2DTranspose(num_filters, (2, 2), strides=2, padding="same")(input)
    x = Concatenate()([x, skip_features])
    x = conv_block(x, num_filters)
    return x

# ═════════════════════════════════════════════════════════════════════════════
# Patch encoder variants
# ═════════════════════════════════════════════════════════════════════════════

def build_patch_encoder(patch_size=(112, 112, 3)):
    """Lightweight CNN patch encoder (no pretrained weights)."""
    inputs = Input(patch_size)
    x = Conv2D(64, 3, activation='relu', padding='same')(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling2D()(x)
    x = Conv2D(128, 3, activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D()(x)
    x = Conv2D(256, 3, activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = GlobalAveragePooling2D()(x)
    return Model(inputs, x, name="PatchEncoder")

def build_patch_encoder1(patch_size=(112, 112, 3)):
    """InceptionResNetV2-based patch encoder with ImageNet weights."""
    inputs = Input(patch_size)
    x = Lambda(preprocess_input)(inputs)
    backbone = InceptionResNetV2(include_top=False, weights="imagenet",
                                  input_tensor=x, pooling="avg")
    return Model(inputs, backbone.output, name="PatchEncoder")

# ─── Residual helpers ─────────────────────────────────────────────────────────

def conv_bn_relu(x, filters, k=3, s=1):
    """Conv2D → BatchNorm → ReLU with configurable kernel and stride."""
    x = Conv2D(filters, k, strides=s, padding='same', use_bias=False)(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    return x

def residual_block(x, filters, k=3):
    """Pre-activation residual block at constant spatial resolution."""
    shortcut = x
    out = conv_bn_relu(x, filters, k=k, s=1)
    out = Conv2D(filters, k, strides=1, padding='same', use_bias=False)(out)
    out = BatchNormalization()(out)
    # Project shortcut if the channel dimension differs
    if shortcut.shape[-1] != filters:
        shortcut = Conv2D(filters, 1, padding='same', use_bias=False)(shortcut)
        shortcut = BatchNormalization()(shortcut)
    out = tf.keras.layers.Add()([out, shortcut])
    out = ReLU()(out)
    return out

def downsample_block(x, filters, blocks=2):
    """Stride-2 spatial downsample followed by `blocks` residual blocks."""
    x = conv_bn_relu(x, filters, k=3, s=2)
    for _ in range(blocks):
        x = residual_block(x, filters, k=3)
    return x

# ═════════════════════════════════════════════════════════════════════════════
# IRv2 patch encoder with multi-scale skip-connection taps
# Input: 112×112  →  Outputs: ps1(112), ps2(56), ps3(28), ps4(14), pb1(7)
# ═════════════════════════════════════════════════════════════════════════════

def build_patch_taps_irv2_112_to_7_simple(weights="imagenet", name="PatchIRv2_112"):
    """
    InceptionResNetV2 encoder that exposes intermediate feature maps at five
    spatial scales for a 112×112 patch input.

    Returns a Keras Model with outputs: [ps1(112), ps2(56), ps3(28), ps4(14), pb1(7)].
    """
    inp = Input((112, 112, 3), name="patch_input_112")
    x = Lambda(preprocess_input, name="irv2_preproc")(inp)
    backbone = InceptionResNetV2(include_top=False, weights=weights, input_tensor=x)

    # Scan all backbone layers and keep the last layer at each target spatial size
    wanted = {112: None, 56: None, 28: None, 14: None, 7: None}
    for layer in backbone.layers:
        shp = getattr(layer, "output_shape", None)
        if not shp or isinstance(shp, list) or len(shp) != 4:
            continue
        h, w = shp[1], shp[2]
        if h == w and h in wanted:
            wanted[h] = layer.output

    missing = [k for k, v in wanted.items() if v is None]
    if missing:
        raise RuntimeError(f"IRv2 didn't expose taps for sizes {missing}. "
                           "Check TF/Keras version or print all layer shapes.")

    ps1 = wanted[112]  # 112×112 early features
    ps2 = wanted[56]   # 56×56
    ps3 = wanted[28]   # 28×28
    ps4 = wanted[14]   # 14×14
    pb1 = wanted[7]    # 7×7 patch bottleneck

    return Model(inp, [ps1, ps2, ps3, ps4, pb1], name=name)

# ═════════════════════════════════════════════════════════════════════════════
# Squeeze-and-Excitation (SE) block
# ═════════════════════════════════════════════════════════════════════════════

def se_block(x, reduction=16):
    """Channel-wise SE recalibration: global pool → FC → sigmoid → scale."""
    filters = K.int_shape(x)[-1]
    se = GlobalAveragePooling2D()(x)
    se = Dense(filters // reduction, activation='relu')(se)
    se = Dense(filters, activation='sigmoid')(se)
    se = Reshape((1, 1, filters))(se)
    return Multiply()([x, se])

# ═════════════════════════════════════════════════════════════════════════════
# Attention-based patch pooling
# ═════════════════════════════════════════════════════════════════════════════

class PatchAttentionPooling(tf.keras.layers.Layer):
    """Soft attention pooling over N patch embeddings → single context vector."""
    def __init__(self, hidden_dim):
        super().__init__()
        self.dense1 = Dense(hidden_dim, activation='tanh')
        self.dense2 = Dense(1)

    def call(self, inputs):
        x       = self.dense1(inputs)              # [B, N, H]
        scores  = self.dense2(x)                   # [B, N, 1]
        weights = tf.nn.softmax(scores, axis=1)
        return tf.reduce_sum(inputs * weights, axis=1)  # [B, D]


class ExtractPatchFeatures(tf.keras.layers.Layer):
    """
    Extracts non-overlapping patches from the full image, encodes each patch
    with `patch_encoder`, then aggregates all patch embeddings via attention pooling.

    Args:
        patch_encoder:   Keras model that maps (patch_h, patch_w, 3) → feature vector.
        patch_size:      Spatial side length of each square patch (pixels).
        stride:          Stride between patch origins (use `patch_size` for non-overlapping tiles).
        attn_hidden_dim: Hidden dimension for the attention pooling MLP.
    """
    def __init__(self, patch_encoder, patch_size, stride, attn_hidden_dim=128, **kwargs):
        super().__init__(**kwargs)
        self.patch_encoder   = patch_encoder
        self.patch_size      = patch_size
        self.stride          = stride
        self.attn_hidden_dim = attn_hidden_dim
        self.attn_pool       = PatchAttentionPooling(attn_hidden_dim)

    def call(self, x):
        # Extract raw patches from the full-resolution image
        patches = tf.image.extract_patches(
            images=x,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.stride, self.stride, 1],
            rates=[1, 1, 1, 1],
            padding='VALID'
        )
        num_patches   = tf.shape(patches)[1] * tf.shape(patches)[2]
        patches       = tf.reshape(patches, (-1, self.patch_size, self.patch_size, 3))

        # Encode each patch independently then aggregate with soft attention
        patch_features = self.patch_encoder(patches)
        patch_features = tf.reshape(patch_features, (-1, num_patches, patch_features.shape[-1]))
        return self.attn_pool(patch_features)  # [B, D]

    def get_config(self):
        config = super().get_config()
        config.update({
            "patch_size": self.patch_size,
            "stride": self.stride,
            "attn_hidden_dim": self.attn_hidden_dim
        })
        return config

    @classmethod
    def from_config(cls, config):
        raise NotImplementedError("Re-instantiate ExtractPatchFeatures with `patch_encoder` manually.")

# ═════════════════════════════════════════════════════════════════════════════
# IRv2 patch encoder with hard-coded skip-tap layer names
# (used inside FoldPatches7to28)
# ═════════════════════════════════════════════════════════════════════════════

L2W = 1e-3  # L2 weight-decay coefficient (tune as needed)

def build_patch_irv2_taps_112_to_7(weights="imagenet", name="PatchIRv2_112_taps"):
    """
    InceptionResNetV2 encoder for 112×112 patches.

    Returns a Model with 5 outputs:
        ps1 (112×112), ps2 (padded ~224), ps3 (padded ~112), ps4 (padded), pb1 (7×7 pool)

    Note: Layer names are pinned to the standard TF2/Keras IRv2 implementation.
          If you upgrade TensorFlow, verify that activation_203/206/276 still exist.
    """
    p_in      = Input((112, 112, 3))
    patch_enc = InceptionResNetV2(include_top=False, weights=weights, input_tensor=p_in)

    # Tap skip connections at specific IRv2 named layers
    ps1 = patch_enc.get_layer("input_2").output
    ps2 = ZeroPadding2D(((1, 0), (1, 0)))(patch_enc.get_layer("activation_203").output)
    ps3 = ZeroPadding2D((1, 1))(patch_enc.get_layer("activation_206").output)
    ps4 = ZeroPadding2D(((2, 1), (2, 1)))(patch_enc.get_layer("activation_276").output)
    pb1 = AveragePooling2D(pool_size=2, strides=2, padding="valid",
                           name="pool_28_to_7")(ps4)

    print("Patch encoder tap shapes:", ps1.shape, ps2.shape, ps3.shape, ps4.shape, pb1.shape)
    return Model(p_in, [ps1, ps2, ps3, ps4, pb1], name=name)


@tf.keras.utils.register_keras_serializable(package="Custom")
class FoldPatches7to28(tf.keras.layers.Layer):
    """
    Splits a 448×448 image into a 4×4 grid of 112×112 patches, runs each patch
    through `per_patch_model` to obtain a 7×7 feature map, then folds all 16 tiles
    back into a single 28×28 spatial feature map.

    This preserves local spatial structure while leveraging patch-level deep
    features from the IRv2 backbone.  The output can be directly concatenated
    with the global encoder's 28×28 bottleneck for feature fusion.

    per_patch_model must return: [ps1, ps2, ps3, ps4, pb1]
    where pb1 has shape (B*16, 7, 7, C).
    """
    def __init__(self, per_patch_model, **kwargs):
        super().__init__(**kwargs)
        self.per_patch_model = per_patch_model
        self._out_channels   = None

    def build(self, input_shape):
        # Try to infer output channels from the model output spec at graph-build time
        try:
            outspec = (self.per_patch_model.output[-1]
                       if isinstance(self.per_patch_model.output, (list, tuple))
                       else self.per_patch_model.output)
            self._out_channels = outspec.shape[-1]
        except Exception:
            self._out_channels = None
        super().build(input_shape)

    def call(self, x):
        # Extract 4×4 non-overlapping 112×112 patches → (B, 4, 4, 112*112*3)
        patches = tf.image.extract_patches(
            images=x, sizes=[1, 112, 112, 1], strides=[1, 112, 112, 1],
            rates=[1, 1, 1, 1], padding="VALID"
        )
        B  = tf.shape(x)[0]
        Gh = tf.shape(patches)[1]  # 4
        Gw = tf.shape(patches)[2]  # 4

        patch_imgs = tf.reshape(patches, (-1, 112, 112, 3))  # (B*16, 112, 112, 3)

        # Run patch encoder; take the last output (7×7 bottleneck)
        outs = self.per_patch_model(patch_imgs)
        y7   = outs[-1] if isinstance(outs, (list, tuple)) else outs  # (B*16, 7, 7, C)

        s = 7
        C = tf.shape(y7)[-1]

        # Fold 16 tiles (4×4 grid of 7×7 maps) → single 28×28 feature map
        y   = tf.reshape(y7, (B, 4, 4, s, s, C))    # (B, 4, 4, 7, 7, C)
        y   = tf.transpose(y, [0, 1, 3, 2, 4, 5])   # (B, 4, 7, 4, 7, C)
        out = tf.reshape(y, (B, 4 * s, 4 * s, C))   # (B, 28, 28, C)
        return out

    def compute_output_shape(self, input_shape):
        b = input_shape[0]
        return (b, 28, 28, self._out_channels)

    def get_config(self):
        return {
            **super().get_config(),
            "per_patch_model": tf.keras.utils.serialize_keras_object(self.per_patch_model),
        }

    @classmethod
    def from_config(cls, config):
        per_patch_model_cfg = config.pop("per_patch_model")
        per_patch_model     = tf.keras.utils.deserialize_keras_object(per_patch_model_cfg)
        return cls(per_patch_model=per_patch_model, **config)

# ═════════════════════════════════════════════════════════════════════════════
# Full Hybrid Segmentation Model
# InceptionResNetV2-UNet + Patch Feature Fusion + SE recalibration
# ═════════════════════════════════════════════════════════════════════════════

def build_inception_resnetv2_unet_with_patch(input_shape, patch_size=112, stride=112):
    """
    Builds the hybrid BML segmentation model.

    Pipeline:
        1. Global IRv2 encoder  → skip features at {448, 224, 112, 56, 28} px.
        2. FoldPatches7to28     → patch-level 28×28 feature map.
        3. Concat + SE block    → channel-recalibrated fused bottleneck.
        4. Four decoder blocks  → progressive upsampling back to 448×448.
        5. Sigmoid output       → binary BML mask.

    Args:
        input_shape: Tuple (H, W, C), expected (448, 448, 3).
        patch_size:  Patch side length in pixels (default 112).
        stride:      Patch extraction stride (default 112 → non-overlapping).
    """
    inputs = Input(input_shape)

    # ── Global encoder (full-image InceptionResNetV2) ─────────────────────
    encoder = InceptionResNetV2(include_top=False, weights="imagenet", input_tensor=inputs)

    # Skip connections at decreasing spatial resolutions
    s1 = encoder.get_layer("input_1").output                                          # 448×448
    s2 = ZeroPadding2D(((1, 0), (1, 0)))(encoder.get_layer("activation").output)     # 224×224
    s3 = ZeroPadding2D((1, 1))(encoder.get_layer("activation_3").output)              # 112×112
    s4 = ZeroPadding2D(((2, 1), (2, 1)))(encoder.get_layer("activation_74").output)  # 56×56
    b1 = ZeroPadding2D((1, 1))(encoder.get_layer("activation_161").output)            # 28×28 bottleneck

    # Project bottleneck to a consistent channel dimension before fusion
    b1_proj = Conv2D(K.int_shape(b1)[-1], 1, activation='relu')(b1)
    b1_proj = BatchNormalization()(b1_proj)

    # ── Patch encoder path ────────────────────────────────────────────────
    patch_model  = build_patch_irv2_taps_112_to_7(weights="imagenet")
    fold_layer   = FoldPatches7to28(patch_model, name="fold_patch_7x7_tiles_to_28x28")
    patch_map_28 = fold_layer(inputs)  # (B, 28, 28, C_patch)

    # ── Feature fusion with SE recalibration ──────────────────────────────
    b1_combined = Concatenate()([b1_proj, patch_map_28])
    b1_combined = se_block(b1_combined)  # channel-wise attention over fused features

    # ── Decoder (progressive upsampling with skip connections) ────────────
    d1 = decoder_block(b1_combined, s4, 512)  # → 56×56
    d2 = decoder_block(d1, s3, 256)           # → 112×112
    d3 = decoder_block(d2, s2, 128)           # → 224×224
    d4 = decoder_block(d3, s1, 64)            # → 448×448

    # ── Output head ───────────────────────────────────────────────────────
    dropout = Dropout(0.5)(d4)
    outputs = Conv2D(1, 1, padding="same", activation="sigmoid")(dropout)

    return Model(inputs, outputs, name="InceptionResNetV2-UNet-PatchFusion")

# ═════════════════════════════════════════════════════════════════════════════
# Loss functions
# ═════════════════════════════════════════════════════════════════════════════

def dice_coef(y_true, y_pred):
    """Hard Dice coefficient (threshold at 0.5) — reported as a metric."""
    y_true_f = K.flatten(y_true)
    y_pred   = K.cast(y_pred, 'float32')
    y_pred_f = K.cast(K.greater(K.flatten(y_pred), 0.5), 'float32')
    intersection = y_true_f * y_pred_f
    return 2. * K.sum(intersection) / (K.sum(y_true_f) + K.sum(y_pred_f))

def dice_coef_loss(y_true, y_pred):
    return -dice_coef(y_true, y_pred)

def dice_loss(y_true, y_pred):
    """Soft Dice loss with Laplace smoothing — used inside composite losses."""
    smooth   = 1.
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = y_true_f * y_pred_f
    return 1. - (2. * K.sum(intersection) + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def tversky_loss(y_true, y_pred, alpha=0.5, beta=0.5):
    """Tversky loss — generalises Dice; alpha/beta control FP vs FN penalty."""
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    TP = K.sum(y_true_f * y_pred_f)
    FP = K.sum((1 - y_true_f) * y_pred_f)
    FN = K.sum(y_true_f * (1 - y_pred_f))
    return 1 - (TP + K.epsilon()) / (TP + alpha * FP + beta * FN + K.epsilon())

def DiceBCELoss(y_true, y_pred):
    """Primary training loss: Binary Cross-Entropy + soft Dice."""
    return binary_crossentropy(y_true, y_pred) + dice_loss(y_true, y_pred)

def bce_logdice_loss(y_true, y_pred):
    return binary_crossentropy(y_true, y_pred) - K.log(1. - dice_loss(y_true, y_pred))

def weighted_bce_loss(y_true, y_pred, weight):
    """Spatially weighted BCE — weight map can emphasise boundary regions."""
    epsilon      = 1e-7
    y_pred       = K.clip(y_pred, epsilon, 1. - epsilon)
    logit_y_pred = K.log(y_pred / (1. - y_pred))
    loss = weight * (logit_y_pred * (1. - y_true) +
                     K.log(1. + K.exp(-K.abs(logit_y_pred))) + K.maximum(-logit_y_pred, 0.))
    return K.sum(loss) / K.sum(weight)

def weighted_bce_dice_loss(y_true, y_pred):
    """Weighted BCE + Dice; weight decays away from the 0.5 mask boundary."""
    y_true = K.cast(y_true, 'float32')
    y_pred = K.cast(y_pred, 'float32')
    averaged_mask = K.pool2d(y_true, pool_size=(50, 50), strides=(1, 1),
                              padding='same', pool_mode='avg')
    weight = K.ones_like(averaged_mask)
    w0     = K.sum(weight)
    weight = 5. * K.exp(-5. * K.abs(averaged_mask - 0.5))
    w1     = K.sum(weight)
    weight *= (w0 / w1)
    return weighted_bce_loss(y_true, y_pred, weight) + dice_loss(y_true, y_pred)

smooth = 1

# ─── Evaluation metrics ───────────────────────────────────────────────────────

def jacard(y_true, y_pred):
    """Jaccard index (Intersection over Union)."""
    y_true_f     = K.flatten(y_true)
    y_pred_f     = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    union        = K.sum(y_true_f + y_pred_f - y_true_f * y_pred_f)
    return intersection / union

def precision_metric(y_true, y_pred):
    """Precision = TP / (TP + FP), thresholded at 0.5."""
    y_true_f            = K.flatten(y_true)
    y_pred_f            = K.cast(K.greater(K.flatten(K.cast(y_pred, 'float32')), 0.5), 'float32')
    true_positives      = K.sum(y_true_f * y_pred_f)
    predicted_positives = K.sum(y_pred_f)
    return true_positives / (predicted_positives + K.epsilon())

def sensitivity_metric(y_true, y_pred):
    """Recall / sensitivity = TP / (TP + FN), thresholded at 0.5."""
    y_true_f         = K.flatten(y_true)
    y_pred_f         = K.cast(K.greater(K.flatten(K.cast(y_pred, 'float32')), 0.5), 'float32')
    true_positives   = K.sum(y_true_f * y_pred_f)
    actual_positives = K.sum(y_true_f)
    return true_positives / (actual_positives + K.epsilon())

## Training & Testing

In [ ]:
import pandas as pd

# ═════════════════════════════════════════════════════════════════════════════
# Data loading utilities
# ═════════════════════════════════════════════════════════════════════════════

def preparadata(traingdata1, traingmask1):
    """
    Loads BMP images and their paired binary masks from disk.

    Processing applied to every sample:
        - BGR → RGB colour conversion.
        - Resize to (height, width) with cubic interpolation.
        - Normalise pixel values to [0, 1].
        - Round mask values to binary {0, 1}.

    Returns:
        X: np.ndarray (N, height, width, 3), float32 in [0, 1].
        Y: np.ndarray (N, height, width, 1), binary {0, 1}.
    """
    img_files = sorted(next(os.walk(traingdata1))[2])
    msk_files = sorted(next(os.walk(traingmask1))[2])
    print(f"Images found: {len(img_files)} | Masks found: {len(msk_files)}")

    X, Y = [], []
    for img_fl in tqdm(img_files):
        if img_fl.split('.')[-1] != 'bmp':
            continue
        img = cv2.imread(os.path.join(traingdata1, img_fl))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        X.append(cv2.resize(img, (height, width), interpolation=cv2.INTER_CUBIC))

        mask_name = img_fl.split('.')[0] + '_mask.bmp'
        msk = cv2.imread(os.path.join(traingmask1, mask_name), cv2.IMREAD_GRAYSCALE)
        Y.append(cv2.resize(msk, (height, width), interpolation=cv2.INTER_CUBIC))

    X = np.array(X) / 255.0
    Y = np.array(Y).reshape((-1, height, width, 1)) / 255.0
    Y = np.round(Y, 0)
    print(f"Dataset shapes — X: {X.shape}, Y: {Y.shape}")
    return X, Y


def preparadataZ(traingdata1, traingmask1):
    """Same as preparadata but also returns image IDs (stem filenames) for output naming."""
    img_files = sorted(next(os.walk(traingdata1))[2])
    msk_files = sorted(next(os.walk(traingmask1))[2])
    print(f"Images found: {len(img_files)} | Masks found: {len(msk_files)}")

    X, Y, Z = [], [], []
    for img_fl in tqdm(img_files):
        if img_fl.split('.')[-1] != 'bmp':
            continue
        img = cv2.imread(os.path.join(traingdata1, img_fl))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        X.append(cv2.resize(img, (height, width), interpolation=cv2.INTER_CUBIC))

        mask_name = img_fl.split('.')[0] + '_mask.bmp'
        msk = cv2.imread(os.path.join(traingmask1, mask_name), cv2.IMREAD_GRAYSCALE)
        Y.append(cv2.resize(msk, (height, width), interpolation=cv2.INTER_CUBIC))
        Z.append(img_fl.split('.')[0])  # store filename stem as image ID

    X = np.array(X) / 255.0
    Y = np.array(Y).reshape((-1, height, width, 1)) / 255.0
    Y = np.round(Y, 0)
    return X, Y, Z


def storeimages(folder, X, Y, Z):
    """Save input, ground-truth mask, and predicted mask PNGs for visual inspection."""
    for i in range(len(Y)):
        plt.figure(figsize=(20, 10))
        plt.imsave(folder + str(i) + '.png',        X[i].squeeze(), cmap='gray')
        plt.imsave(folder + str(i) + '_ground.png', Y[i].squeeze(), cmap='magma')
        plt.imsave(folder + str(i) + '_pred.png',   Z[i].squeeze(), cmap='magma')
        plt.close()


if __name__ == '__main__':

    # ── Load training and validation data ─────────────────────────────────
    X,  Y  = preparadata(traingdata, traingmask)
    X1, Y1 = preparadata(validdata,  validmask)

    # ── Training callbacks ─────────────────────────────────────────────────
    reduce_lr      = ReduceLROnPlateau(monitor='val_loss', factor=0.1,
                                        patience=75, verbose=1, min_lr=1e-6, mode='min')
    early_stopping = EarlyStopping(monitor='val_loss', min_delta=0, verbose=1,
                                    patience=75, mode='min', restore_best_weights=True)

    # ── Build and compile the model ────────────────────────────────────────
    keras.backend.clear_session()
    opt   = AdamW(learning_rate=1e-3, weight_decay=1e-4)
    model = build_inception_resnetv2_unet_with_patch((height, width, 3))
    model.compile(
        optimizer=opt,
        loss=DiceBCELoss,
        metrics=[jacard, dice_coef, precision_metric, sensitivity_metric]
    )

    # ── Train ──────────────────────────────────────────────────────────────
    new_model = model.fit(
        X, Y,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        verbose=1,
        shuffle=False,
        validation_data=(X1, Y1),
        callbacks=[early_stopping, reduce_lr]
    )
    print("Training history keys:", new_model.history.keys())

    # ── Load test data and generate predictions ────────────────────────────
    X2, Y2, Z2 = preparadataZ(testingdata, testingmask)
    yp = model.predict(x=X2, batch_size=BATCH_SIZE, verbose=1)
    yp = np.round(yp, 0)  # binarise at 0.5 threshold

    # ── Save prediction images ─────────────────────────────────────────────
    # TODO: Update output_img_dir to your preferred predictions output folder
    output_img_dir = "<PATH_TO_YOUR_OUTPUT>/predictions/"
    os.makedirs(output_img_dir, exist_ok=True)

    for image, image_id in zip(yp, Z2):
        image_uint8 = (image[:, :, 0] * 255.).astype(np.uint8)
        cv2.imwrite(os.path.join(output_img_dir, f"{image_id}_pred.jpg"), image_uint8)

    # ═════════════════════════════════════════════════════════════════════
    # Per-image and patient-level Dice computation
    # ═════════════════════════════════════════════════════════════════════

    def dice_score_for_reporting(y_true, y_pred):
        """Compute hard Dice for a single sample (wraps keras metric as numpy scalar)."""
        y_true_tf = tf.convert_to_tensor(y_true, dtype=tf.float32)
        y_pred_tf = tf.convert_to_tensor(y_pred, dtype=tf.float32)
        return float(dice_coef(y_true_tf, y_pred_tf))

    def get_patient_id(image_id):
        """Extract patient ID from filename stem (handles '_v00' and '_' delimiters)."""
        image_id = os.path.splitext(os.path.basename(str(image_id)))[0]
        if "_v00" in image_id:
            return image_id.split("_v00")[0]
        return image_id.split("_")[0]

    rows, patient_data = [], {}
    for i in range(len(yp)):
        fname     = str(Z2[i])
        pid       = get_patient_id(fname)
        y_true_np = (np.asarray(Y2[i]).squeeze() > 0.5).astype(np.float32)
        y_pred_np = (np.asarray(yp[i]).squeeze() > 0.5).astype(np.float32)
        dice_i    = dice_score_for_reporting(Y2[i], yp[i])

        rows.append({"image_id": fname, "Patient_ID": pid, "dice": dice_i})
        patient_data.setdefault(pid, {"gt": [], "pred": [], "dice_2d_scores": []})
        patient_data[pid]["gt"].append(y_true_np)
        patient_data[pid]["pred"].append(y_pred_np)
        patient_data[pid]["dice_2d_scores"].append(dice_i)

    # ── Aggregate to patient-level volumetric metrics ──────────────────────
    results, x_volumes, y_volumes = [], [], []
    for pid, data in patient_data.items():
        vol_pred = np.array(data["pred"], dtype=np.float32)
        vol_gt   = np.array(data["gt"],   dtype=np.float32)

        pred_vol = np.sum(vol_pred)
        gt_vol   = np.sum(vol_gt)
        x_volumes.append(pred_vol)
        y_volumes.append(gt_vol)

        avg_2d_dice    = np.mean(data["dice_2d_scores"])
        intersection3d = np.sum(vol_pred * vol_gt)
        denom3d        = pred_vol + gt_vol
        dice_3d        = (2.0 * intersection3d + 1e-7) / (denom3d + 1e-7)

        results.append({
            "Patient_ID": pid,
            "Slices_Count": len(vol_pred),
            "Avg_2D_Dice": float(avg_2d_dice),
            "Volumetric_3D_Dice": float(dice_3d),
            "Predicted_Volume_Voxels": float(pred_vol),
            "Ground_Truth_Volume_Voxels": float(gt_vol),
        })

    per_image_df       = pd.DataFrame(rows)
    patient_metrics_df = pd.DataFrame(results).sort_values("Patient_ID").reset_index(drop=True)

    # ── Save CSV reports ───────────────────────────────────────────────────
    # TODO: Update out_dir to your preferred results output folder
    out_dir = "<PATH_TO_YOUR_OUTPUT>/results"
    os.makedirs(out_dir, exist_ok=True)
    csv_path         = os.path.join(out_dir, "per_image_dice.csv")
    patient_csv_path = os.path.join(out_dir, "patient_wise_metrics.csv")
    per_image_df.to_csv(csv_path, index=False)
    patient_metrics_df.to_csv(patient_csv_path, index=False)

    for _, row in per_image_df.iterrows():
        print(row["image_id"], row["dice"])
    print(f"Per-image dice saved to:  {csv_path}")
    print(f"Patient-wise metrics to:  {patient_csv_path}")
    display(patient_metrics_df)

    # ── Final evaluation on the test set ───────────────────────────────────
    score = model.evaluate(X2, Y2, batch_size=BATCH_SIZE)

    precision_final   = score[3] * 100
    recall_final      = score[4] * 100
    iou_final         = score[1] * 100
    avg_2d_dice_final = patient_metrics_df["Avg_2D_Dice"].mean() * 100
    avg_3d_dice_final = patient_metrics_df["Volumetric_3D_Dice"].mean() * 100

    # ── Pearson correlation between predicted and ground-truth volumes ─────
    # Voxel spacing: 0.5 mm × 0.5 mm × 3.0 mm — adjust to your scanner protocol
    voxel_vol_mm3 = 0.5 * 0.5 * 3.0

    if len(x_volumes) > 1:
        x_arr = np.array(x_volumes, dtype=np.float64)
        y_arr = np.array(y_volumes, dtype=np.float64)
        r_vox = np.corrcoef(x_arr, y_arr)[0, 1]

        x_mm = x_arr * voxel_vol_mm3
        y_mm = y_arr * voxel_vol_mm3
        r_mm = np.corrcoef(x_mm, y_mm)[0, 1]

        patient_metrics_df["Predicted_Volume_mm3"]    = x_mm
        patient_metrics_df["Ground_Truth_Volume_mm3"] = y_mm
        patient_metrics_df.to_csv(patient_csv_path, index=False)
    else:
        r_vox = np.nan
        r_mm  = np.nan

    print("\n=== Final Evaluation Metrics ===")
    print(f"Precision:                            {precision_final:.4f}%")
    print(f"Recall (Sensitivity):                 {recall_final:.4f}%")
    print(f"IoU / Jaccard:                        {iou_final:.4f}%")
    print(f"Avg 2D Dice (per patient):            {avg_2d_dice_final:.4f}%")
    print(f"Avg 3D Volumetric Dice (per patient): {avg_3d_dice_final:.4f}%")

    if len(x_volumes) > 1:
        print(f"Pearson Correlation (Voxels): {r_vox:.4f}")
        print(f"Pearson Correlation (mm³):    {r_mm:.4f}")
    else:
        print("Need > 1 patient to compute Pearson correlation.")